# Test the live RAG pipeline

Tests `src/derm/rag/answer.py` end-to-end: retrieval → LLM → cited answer. Runs against the existing Chroma index at `data/public/kb_chroma/` (built by `03_build_rag_cache.ipynb` — re-run that notebook if the KB markdown files change).

Two providers: local `llama3.1:8b` via Ollama (free, slower) or OpenAI `gpt-4o-mini` (paid, faster). Switch with the `LLM_PROVIDER` env var.

## Step 1 — Setup

Make `src/` importable from the notebook's `notebooks/` working directory.

In [ ]:
# Claude-assisted with this import
import os
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from derm.rag.retriever import retrieve
from derm.rag.answer import answer

In [ ]:
#%pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openai]2m3/4 [openai]
Note: you may need to restart the kernel to use updated packages.


## Step 2 — Sanity-check retrieval

Are the right chunks coming back? If a BCC question pulls SK chunks, embedding is broken — fix that before judging the LLM.

In [ ]:
queries = [
    "How can I tell SCC in situ from a seborrheic keratosis?",
    "Why is mycosis fungoides commonly misdiagnosed early on?",
    "Which conditions present differently in darker skin tones?",
]

for q in queries:
    print(f"\nQ: {q}")
    for c in retrieve(q, k=3):
        print(f"  \u2022 [{c['title']} / {c['section']}]  ({c['source']})")


Q: How can I tell SCC in situ from a seborrheic keratosis?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  • [Seborrheic keratosis (brown warts, basal cell papillomas) / Diagnosis]  (DermNet NZ)
  • [Seborrheic keratosis (brown warts, basal cell papillomas) / Risk factors and associations]  (DermNet NZ)
  • [Basal cell carcinoma / Key distinguishing features]  (DermNet NZ)

Q: Why is mycosis fungoides commonly misdiagnosed early on?
  • [Mycosis fungoides / Causes]  (DermNet NZ)
  • [Mycosis fungoides / Diagnosis]  (DermNet NZ)
  • [Mycosis fungoides / Risk factors and prognosis]  (DermNet NZ)

Q: Which conditions present differently in darker skin tones?
  • [Basal cell carcinoma / Variation in skin tones]  (DermNet NZ)
  • [Moles (melanocytic nevi, pigmented nevi) / Variation in skin tones]  (DermNet NZ)
  • [Viral wart — extra information / Variation in skin tones]  (DermNet NZ)


## Step 3 — End-to-end with Ollama (local, free)

Four representative queries spanning single-condition, comparison, and broad/no-condition scenarios. Eyeball each: are sources scoped right? Are citations inline per-claim? Do bullets use `**<feature label>**: prose` format?

In [ ]:
# claude only assisted setting up env variable and printing dividers
os.environ["LLM_PROVIDER"] = "ollama"

queries = [
    "How can I tell SCC in situ from a seborrheic keratosis?",
    "What is basal cell carcinoma?",
    "How can I tell verruca vulgaris from seborrheic keratosis?",
    "How does skin tone affect dermatology presentation?",
]

for q in queries:
    print("=" * 70)
    print(f"Q: {q}")
    print("=" * 70)
    result = answer(q)
    print(result["content"])
    print("\n--- Sources ---")
    for i, s in enumerate(result["sources"], 1):
        print(f"[{i}] {s['source']} — {s['title']}")
    print()

Q: How can I tell SCC in situ from a seborrheic keratosis?
To distinguish squamous cell carcinoma in situ (SCC in situ) from a seborrheic keratosis (SK), consider the following features:

• **Border characteristics**: SCC in situ often has sharp, irregular borders [1], which contrasts with the smooth, sharply demarcated borders of SK [2].
• **Color and texture**: SCC in situ typically presents as a persistent, slow-growing scaly plaque with an orange-red or brown color [1], whereas SK is usually a dull tan/brown warty plaque [2].

These differences can help you distinguish between the two conditions. However, it's essential to note that dermoscopy may be necessary for a definitive diagnosis, especially if there's any doubt about the lesion's nature.

--- Sources ---
[1] DermNet NZ — Squamous cell carcinoma in situ (intraepidermal squamous cell carcinoma)
[2] DermNet NZ — Seborrheic keratosis (brown warts, basal cell papillomas)

Q: What is basal cell carcinoma?
Basal cell carcinoma (BC

## Step 4 — End-to-end with OpenAI (paid, faster)

Set `OPENAI_API_KEY` in your shell first (e.g. `export OPENAI_API_KEY=...`), then uncomment.

In [ ]:
# claude only assisted setting up env variable and printing dividers
os.environ["LLM_PROVIDER"] = "openai"

queries = [
    "How can I tell SCC in situ from a seborrheic keratosis?",
    "What is basal cell carcinoma?",
    "How can I tell verruca vulgaris from seborrheic keratosis?",
    "How does skin tone affect dermatology presentation?",
]

for q in queries:
    print("=" * 70)
    print(f"Q: {q}")
    print("=" * 70)
    result = answer(q)
    print(result["content"])
    print("\n--- Sources ---")
    for i, s in enumerate(result["sources"], 1):
        print(f"[{i}] {s['source']} — {s['title']}")
    print()

Q: How can I tell SCC in situ from a seborrheic keratosis?
To differentiate squamous cell carcinoma in situ from seborrheic keratosis, several key features can be compared.

• **Color and texture**: Squamous cell carcinoma in situ often presents as a persistent, slow-growing scaly plaque with an orange-red or brown color [1], while seborrheic keratosis typically appears as a stuck-on, well-demarcated warty plaque that is dull tan or brown [2].

• **Border characteristics**: The borders of squamous cell carcinoma in situ are sharp and irregular [1], contrasting with the smooth, sharply demarcated borders of seborrheic keratosis, which are well-defined [2].

• **Response to treatment**: Squamous cell carcinoma in situ does not respond to topical steroids and persists over months to years [1], whereas seborrheic keratosis is stable in size and color over time and may not require treatment unless there is sudden change [2].

• **Dermoscopy findings**: On dermoscopy, squamous cell carcinoma

## Step 5 — Tuning notes

Current knobs (after iteration):

**`src/derm/rag/answer.py`:**
- `TOP_K = 5` — max chunks fed to the LLM. Dedup by URL + filter-to-cited collapses the source list, so the rendered tooltip count is usually 1–3.
- `temperature = 0` for both providers — fully deterministic, citation-grounded.
- `OPENAI_MODEL = "gpt-4o-mini"` — swap to `gpt-4o` for higher quality at higher cost.
- `OLLAMA_MODEL = "llama3.1:8b"` — local, free, slightly weaker at per-claim citation discipline than OpenAI.
- `SYSTEM_PROMPT` — controls citation placement (inline-per-claim) and bullet structure (**Label**: prose). Edit if format drifts.

**`src/derm/rag/retriever.py`:**
- `lambda_mult = 0.7` — MMR balance: relevance (1.0) ↔ diversity (0.0). Lower for broader cross-condition retrieval; higher if off-topic noise creeps in.
- `fetch_k = 20` — MMR candidate pool size before diversity reranking.
- `CONDITION_KEYWORDS` — query→condition routing. Add aliases here if a phrasing doesn't match the right article.